In [1]:
import re
import torch
import numpy as np
import plotly.graph_objects as go
from transformers import BertTokenizer, BertModel
from sklearn.cluster import KMeans
from umap import UMAP

In [2]:
tokenizer = BertTokenizer.from_pretrained('l3cube-pune/kannada-sentence-similarity-sbert') 
model = BertModel.from_pretrained('l3cube-pune/kannada-sentence-similarity-sbert')

In [3]:
def get_sentences(file_path, word):
    """ Extract sentences containing the specified word exactly, handling newline characters. """
    # Regex for exact word match considering Unicode properties for Kannada
    word_pattern = re.compile(r'(?<![\u0C80-\u0CFF])' + re.escape(word) + r'(?![\u0C80-\u0CFF])', re.IGNORECASE)
    matching_sentences = []
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        sentences = re.split(r'[.!?]|\n', content)  # Split on punctuation or newlines
        for sentence in sentences:
            sentence = sentence.strip()
            if word_pattern.search(sentence):
                matching_sentences.append(sentence)
    return matching_sentences

def embed_sentences(sentences):
    """ Embeds sentences using BERT. """
    embeddings = []
    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state[:, 0, :])
    return torch.cat(embeddings, dim=0)

def visualize_embeddings_with_umap(embeddings, sentences, n_clusters=5):
    """ Projects embeddings to 2D using UMAP and creates an interactive scatter plot. """
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)

    reducer = UMAP(n_components=2, random_state=42)
    embeddings_2d = reducer.fit_transform(embeddings)

    fig = go.Figure()
    for i in range(n_clusters):
        cluster_indices = labels == i
        cluster_embeddings = embeddings_2d[cluster_indices]
        cluster_sentences = np.array(sentences)[cluster_indices]
        hover_text = [f"{sent}" for sent in cluster_sentences]
        fig.add_trace(go.Scatter(x=cluster_embeddings[:, 0], y=cluster_embeddings[:, 1], mode='markers',
                                 marker=dict(size=10), text=hover_text, hoverinfo="text", name=f'Cluster {i+1}'))

    fig.update_layout(title=f'UMAP Projection of Sentence Embeddings', hovermode='closest',
                      xaxis_title='Dimension 1', yaxis_title='Dimension 2')
    fig.show()

In [4]:
file_path = 'sentences.txt'
word = 'ಮುತ್ತು'
sentences = get_sentences(file_path, word)
embeddings = embed_sentences(sentences)
visualize_embeddings_with_umap(embeddings, sentences, n_clusters=2)

c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-to-text", model="DunnBC22/trocr-base-printed_license_plates_ocr")

KeyError: <class 'transformers.models.vision_encoder_decoder.configuration_vision_encoder_decoder.VisionEncoderDecoderConfig'>

In [ ]:
pipe('https://stage-drupal.car.co.uk/s3fs-public/styles/original_size/public/2019-09/why-are-number-plates-yellow-and-white.jpg?rt1UJUyIi7L2DpS613hFYlI5ng3U4QT3&itok=3SZjXU0B')